# 02 — Model Training
**Project:** Casting Defect Detection

**Goal of this notebook:** train a simple baseline CNN, then compare three transfer-learning models,
and save the best one for evaluation and Grad-CAM in `03_evaluation_gradcam.ipynb`.

| Model | Why it is included |
|---|---|
| Baseline CNN | Reference point: how far a small model trained from scratch can go |
| MobileNetV2 | Lightweight and fast, suitable for edge devices next to a production line |
| EfficientNet-B0 | Strong accuracy for its size |
| ResNet50V2 | Deeper, classic architecture as a heavier comparison |

Decisions carried over from the EDA:
- **Defective = positive class (label 1)**, because missing a defect is the costly error
- **Class weights** to handle the mild imbalance
- **Recall on the defective class** is tracked during training
- Images resized to **224×224** and loaded as **3-channel RGB**

> Run on **Google Colab with a GPU**: Runtime → Change runtime type → **T4 GPU** → Run all.
> Expected time: roughly 30–60 minutes for all four models.

## 1. Setup

In [ ]:
import os
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow, switch to a GPU runtime)")

FIG_DIR = "reports/figures"
MODEL_DIR = "models"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## 2. Configuration
All settings in one place, so experiments are easy to repeat and change.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.15          # 15% of the training data is held out for validation

EPOCHS_BASELINE = 15
EPOCHS_FROZEN = 8         # transfer learning, phase 1: train only the new head
EPOCHS_FINETUNE = 6       # transfer learning, phase 2: unfreeze the top of the backbone
FINETUNE_LAYERS = 30      # how many top backbone layers to unfreeze in phase 2

WEIGHTS = "imagenet"      # pretrained weights for the backbones

## 3. Download and locate the dataset

In [ ]:
!pip install -q kagglehub
import kagglehub

DATA_ROOT = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")
print("Downloaded to:", DATA_ROOT)

In [ ]:
def find_split_root(root):
    """Return the folder that contains both 'train' and 'test' with class subfolders."""
    for dirpath, dirnames, _ in os.walk(root):
        if "train" in dirnames and "test" in dirnames:
            train_dir = os.path.join(dirpath, "train")
            if {"def_front", "ok_front"} <= set(os.listdir(train_dir)):
                return dirpath
    raise FileNotFoundError("Could not find train/test folders with def_front and ok_front")

SPLIT_ROOT = find_split_root(DATA_ROOT)
TRAIN_DIR = os.path.join(SPLIT_ROOT, "train")
TEST_DIR = os.path.join(SPLIT_ROOT, "test")
print("Train:", TRAIN_DIR)
print("Test: ", TEST_DIR)

## 4. Load the images
Keras sorts class folders alphabetically, so it assigns `def_front = 0` and `ok_front = 1`.
We **flip the labels** so that **defective = 1** (the positive class). This way, "recall" and
"precision" in the metrics refer to the defective class.

The test set is kept completely separate and is **only used once, at the end**.

In [ ]:
CLASS_NAMES = ["def_front", "ok_front"]

def load(directory, subset=None, shuffle=True):
    return keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        color_mode="rgb",          # grayscale images are converted to 3 channels
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
        validation_split=VAL_SPLIT if subset else None,
        subset=subset,
    )

train_raw = load(TRAIN_DIR, subset="training")
val_raw = load(TRAIN_DIR, subset="validation")
test_raw = load(TEST_DIR, shuffle=False)

def to_defect_label(images, labels):
    # def_front (0) -> 1 = defective, ok_front (1) -> 0 = ok
    return images, tf.cast(1 - labels, tf.float32)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(to_defect_label).cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_raw.map(to_defect_label).cache().prefetch(AUTOTUNE)
test_ds = test_raw.map(to_defect_label).cache().prefetch(AUTOTUNE)

## 5. Class weights
The defective class is slightly larger. Class weights make each mistake on the smaller class
count a bit more, so the model does not simply favor the majority class.

In [ ]:
train_labels = np.concatenate([y.numpy() for _, y in train_ds]).astype(int)
n_total = len(train_labels)
n_def = int(train_labels.sum())
n_ok = n_total - n_def

class_weight = {
    0: n_total / (2 * n_ok),    # ok
    1: n_total / (2 * n_def),   # defective
}
print(f"Training images: {n_total}  (defective: {n_def}, ok: {n_ok})")
print("Class weights:", {k: round(v, 3) for k, v in class_weight.items()})

## 6. Data augmentation
Small random changes during training simulate variation on a real line (part orientation,
lighting) and reduce overfitting. Augmentation is only active during training.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.05),       # small zoom only, so the rims are not cut off
    layers.RandomContrast(0.1),
], name="augmentation")

# Preview the augmentation on one training image
images, labels = next(iter(train_ds))
plt.figure(figsize=(10, 2.5))
for i in range(5):
    ax = plt.subplot(1, 5, i + 1)
    aug = augment(images[:1], training=True)[0]
    ax.imshow(aug.numpy().astype("uint8"))
    ax.axis("off")
plt.suptitle("Augmented versions of one training image")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/05_augmentation.png", dpi=150)
plt.show()

## 7. Metrics and training helpers

In [ ]:
def make_metrics():
    return [
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ]

def make_callbacks(name):
    return [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(f"{MODEL_DIR}/{name}.keras", monitor="val_loss", save_best_only=True),
    ]

def merge_histories(*histories):
    merged = {}
    for h in histories:
        for k, v in h.history.items():
            merged.setdefault(k, []).extend(v)
    return merged

HISTORIES = {}

## 8. Baseline CNN (trained from scratch)
A small network with three convolution blocks. Its score tells us how much transfer
learning actually adds.

In [ ]:
def build_baseline():
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = layers.Rescaling(1.0 / 255)(x)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

baseline = build_baseline()
baseline.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=make_metrics())
baseline.summary()

h = baseline.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_BASELINE,
    class_weight=class_weight, callbacks=make_callbacks("baseline_cnn"),
)
HISTORIES["baseline_cnn"] = merge_histories(h)

## 9. Transfer learning
Each pretrained backbone was trained on ImageNet (1.2M everyday photos). We reuse the features
it already learned (edges, textures, shapes) and train a new classification head for OK/defective.

Training happens in **two phases**:
1. **Frozen:** the backbone is locked; only the new head is trained (fast, stable).
2. **Fine-tuning:** the top layers of the backbone are unlocked and trained with a
   **very small learning rate**, so they adapt to casting images without forgetting what they learned.

Each backbone expects its input scaled in a specific way, so a matching preprocessing layer is
built into the model. This means the saved model accepts **raw images (0–255)** directly, which
keeps the demo app simple.

In [ ]:
BACKBONES = {
    # name: (constructor, preprocessing layer)
    "mobilenetv2": (keras.applications.MobileNetV2, layers.Rescaling(1 / 127.5, offset=-1)),   # [-1, 1]
    "efficientnetb0": (keras.applications.EfficientNetB0, None),                               # scales internally
    "resnet50v2": (keras.applications.ResNet50V2, layers.Rescaling(1 / 127.5, offset=-1)),     # [-1, 1]
}

def build_transfer(name):
    constructor, preprocess = BACKBONES[name]
    base = constructor(include_top=False, weights=WEIGHTS, input_shape=IMG_SIZE + (3,))
    base.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    if preprocess is not None:
        x = preprocess(x)
    x = base(x, training=False)    # keep BatchNorm layers in inference mode, also during fine-tuning
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name=name), base

def train_transfer(name):
    print(f"\n========== {name} ==========")
    model, base = build_transfer(name)

    # Phase 1: frozen backbone
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=make_metrics())
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FROZEN,
                   class_weight=class_weight, callbacks=make_callbacks(name))

    # Phase 2: fine-tune the top layers (BatchNorm layers stay frozen)
    base.trainable = True
    for layer in base.layers[:-FINETUNE_LAYERS]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=make_metrics())
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
                   class_weight=class_weight, callbacks=make_callbacks(name))

    HISTORIES[name] = merge_histories(h1, h2)
    return model

In [ ]:
for name in BACKBONES:
    train_transfer(name)

## 10. Training curves

In [ ]:
fig, axes = plt.subplots(len(HISTORIES), 2, figsize=(12, 3.2 * len(HISTORIES)), squeeze=False)
for row, (name, hist) in enumerate(HISTORIES.items()):
    axes[row, 0].plot(hist["loss"], label="train")
    axes[row, 0].plot(hist["val_loss"], label="validation")
    axes[row, 0].set_title(f"{name} — loss")
    axes[row, 1].plot(hist["recall"], label="train")
    axes[row, 1].plot(hist["val_recall"], label="validation")
    axes[row, 1].set_title(f"{name} — defective recall")
    for ax in axes[row]:
        ax.set_xlabel("epoch")
        ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/06_training_curves.png", dpi=150)
plt.show()

**How to read this:** if training loss keeps falling while validation loss rises, the model is
**overfitting**. For transfer models, the jump in the middle of the curve is where fine-tuning starts.

## 11. Compare all models on the test set
Each model's best checkpoint (lowest validation loss) is loaded and evaluated **once** on the
untouched test set, using the default threshold of 0.5. Threshold tuning and the confusion matrix
are done in notebook 03.

We also measure **inference time per image**, which matters on a production line.

In [ ]:
y_test = np.concatenate([y.numpy() for _, y in test_ds]).astype(int)
results = []

for name in HISTORIES:
    model = keras.models.load_model(f"{MODEL_DIR}/{name}.keras")
    metrics = model.evaluate(test_ds, return_dict=True, verbose=0)

    # Inference time: average over the whole test set (after one warm-up batch)
    model.predict(test_ds.take(1), verbose=0)
    start = time.perf_counter()
    model.predict(test_ds, verbose=0)
    ms_per_image = (time.perf_counter() - start) / len(y_test) * 1000

    p, r = metrics["precision"], metrics["recall"]
    results.append({
        "model": name,
        "accuracy": metrics["accuracy"],
        "precision": p,
        "recall (defective)": r,
        "f1": 2 * p * r / (p + r + 1e-9),
        "auc": metrics["auc"],
        "ms_per_image": ms_per_image,
        "params (M)": model.count_params() / 1e6,
    })

comparison = pd.DataFrame(results).set_index("model").round(4)
comparison = comparison.sort_values(["f1", "recall (defective)"], ascending=False)
os.makedirs("reports", exist_ok=True)
comparison.to_csv("reports/model_comparison.csv")
comparison

In [ ]:
ax = comparison[["accuracy", "precision", "recall (defective)", "f1"]].plot(kind="bar", figsize=(9, 4), rot=0)
ax.set_ylim(0, 1.05)
ax.set_title("Model comparison on the test set (threshold 0.5)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/07_model_comparison.png", dpi=150)
plt.show()

## 12. Select the best model
We want the model that **misses the fewest defects**, but recall alone can be fooled: a model that
labels *everything* as defective gets 100% recall and is useless. So the rule is:
1. keep only models whose **F1 is within 0.01 of the best F1** (good overall balance), then
2. among those, pick the one with the **highest defective recall**.

In [ ]:
best_f1 = comparison["f1"].max()
candidates = comparison[comparison["f1"] >= best_f1 - 0.01]
best_name = candidates.sort_values(["recall (defective)", "f1"], ascending=False).index[0]
# Each model is already saved under its own name, e.g. models/resnet50v2.keras.
# Record which one was selected, so notebook 03 and the app know which file to use.
with open(f"{MODEL_DIR}/selected_model.txt", "w") as f:
    f.write(best_name)

print("Selected model:", best_name, f"-> {MODEL_DIR}/{best_name}.keras")
print(comparison.loc[best_name])

In [ ]:
# Download the results to your computer (Colab only)
!zip -qr training_outputs.zip models reports
try:
    from google.colab import files
    files.download("training_outputs.zip")
except ImportError:
    print("Not running on Colab: outputs are in the models/ and reports/ folders")

## 13. Summary

| Model | Accuracy | Precision | Recall (defective) | F1 | Missed defects | False alarms | ms / image | Params (M) |
|---|---|---|---|---|---|---|---|---|
| **ResNet50V2** | 99.58% | 99.56% | **99.78%** | 99.67% | **1** | 2 | 5.19 | 23.57 |
| MobileNetV2 | 99.58% | 100% | 99.34% | 99.67% | 3 | 0 | 2.87 | 2.26 |
| EfficientNet-B0 | 99.02% | 100% | 98.45% | 99.22% | 7 | 0 | 4.78 | 4.05 |
| Baseline CNN | 83.92% | 83.01% | 93.82% | 88.08% | 28 | 87 | 0.57 | 0.09 |

- **Transfer learning makes a large difference:** all pretrained models reached about 99% accuracy versus 83.9% for the baseline CNN.
- **Best model: ResNet50V2.** It tied with MobileNetV2 on accuracy and F1 but had the highest defective recall, missing only 1 of 453 defective parts.
- **Deployment alternative: MobileNetV2.** Two more missed defects but zero false alarms, about 10× smaller and 1.8× faster, attractive for an edge device on the line.

> **Note on very high scores:** this dataset is clean and augmented by its author, so similar images may appear in both train and test. Real production performance must be validated with new data.

**Next:** `03_evaluation_gradcam.ipynb` — confusion matrix, threshold tuning, and Grad-CAM heatmaps.